In [1]:
import pandas as pd
import pyodbc
import json
from datetime import date, datetime

# Charger configuration JSON
with open("staging_config.json", encoding="utf-8") as f:
    config_list = json.load(f)

# Connexion SQL Server (adapter)
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=Staging_Test;"
    "UID=moeness;"
    "PWD=azerty"
)


In [2]:
# Dictionnaire des tables référentielles
referentiels = {
    "Postes": pd.read_sql("SELECT * FROM Postes", conn),
    "Categories": pd.read_sql("SELECT * FROM Categories", conn),
    "Pieds": pd.read_sql("SELECT * FROM Pieds", conn),
    "Competitions": pd.read_sql("SELECT * FROM Competitions", conn),
    "Meteo": pd.read_sql("SELECT * FROM Meteo", conn),
    "Etats_Terrain": pd.read_sql("SELECT * FROM Etats_Terrain", conn),
    "Resultats": pd.read_sql("SELECT * FROM Resultats", conn)
}


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\3942123766.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  "Postes": pd.read_sql("SELECT * FROM Postes", conn),
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\3942123766.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  "Categories": pd.read_sql("SELECT * FROM Categories", conn),
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\3942123766.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  "Pieds": pd.read_sql("SELECT * FROM Pieds", conn),
C:\Users\MOEµNESS\A

In [3]:
def load_and_historize(conn, config):
    table = config["table"]
    csv_path = config["csv"]
    id_col = config["id_col"]
    compare_cols = config["compare_cols"].copy()  # important pour ne pas modifier l'original
    today = date.today()

    try:
        df = pd.read_csv(csv_path, parse_dates=["Date_Examen"] if "Date_Examen" in compare_cols else [])
    except Exception as e:
        print(f"❌ Erreur lecture CSV {csv_path} : {e}")
        return

    df = df.astype("object")

    # 🔁 Mapping automatique selon la table
    if table == "Joueurs":
        if "Position" in compare_cols:
            df = df.merge(referentiels["Postes"], how="left", left_on="Position", right_on="Nom_Poste")
            df["ID_Poste"] = df["ID_Poste"].astype(int)
            compare_cols.remove("Position")
            compare_cols.append("ID_Poste")

        if "Catégorie" in compare_cols:
            df = df.merge(referentiels["Categories"], how="left", left_on="Catégorie", right_on="Nom_Categorie")
            df["ID_Categorie"] = df["ID_Categorie"].astype(int)
            compare_cols.remove("Catégorie")
            compare_cols.append("ID_Categorie")

        if "Pied_Dominant" in compare_cols:
            df = df.merge(referentiels["Pieds"], how="left", left_on="Pied_Dominant", right_on="Cote_Pied")
            df["ID_Pied"] = df["ID_Pied"].astype(int)
            compare_cols.remove("Pied_Dominant")
            compare_cols.append("ID_Pied")

    elif table == "Matchs":
        if "Compétition" in compare_cols:
            df = df.merge(referentiels["Competitions"], how="left", left_on="Compétition", right_on="Nom_Competition")
            df["ID_Competition"] = df["ID_Competition"].astype(int)
            compare_cols.remove("Compétition")
            compare_cols.append("ID_Competition")

        if "Résultat" in compare_cols:
            df = df.merge(referentiels["Resultats"], how="left", left_on="Résultat", right_on="Nom_Resultat")
            df["ID_Resultat"] = df["ID_Resultat"].astype(int)
            compare_cols.remove("Résultat")
            compare_cols.append("ID_Resultat")

        if "Catégorie" in compare_cols:
            df = df.merge(referentiels["Categories"], how="left", left_on="Catégorie", right_on="Nom_Categorie")
            df["ID_Categorie"] = df["ID_Categorie"].astype(int)
            compare_cols.remove("Catégorie")
            compare_cols.append("ID_Categorie")

    elif table == "Donnees_Contextuelles":
        if "Météo" in compare_cols:
            df = df.merge(referentiels["Meteo"], how="left", left_on="Météo", right_on="Type_Meteo")
            df["ID_Meteo"] = df["ID_Meteo"].astype(int)
            compare_cols.remove("Météo")
            compare_cols.append("ID_Meteo")

        if "Etat_Terrain" in compare_cols:
            df = df.merge(referentiels["Etats_Terrain"], how="left", left_on="Etat_Terrain", right_on="Etat")
            df["ID_Etat"] = df["ID_Etat"].astype(int)
            compare_cols.remove("Etat_Terrain")
            compare_cols.append("ID_Etat")

        if "Compétition" in compare_cols:
            df = df.merge(referentiels["Competitions"], how="left", left_on="Compétition", right_on="Nom_Competition")
            df["ID_Competition"] = df["ID_Competition"].astype(int)
            compare_cols.remove("Compétition")
            compare_cols.append("ID_Competition")

    try:
        df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)
    except Exception as e:
        print(f"❌ Erreur lecture table {table} : {e}")
        return

    cursor = conn.cursor()

    for _, row in df.iterrows():
        old = df_db[df_db[id_col] == row[id_col]]
        if old.empty:
            # 🔵 Nouvelle ligne
            cols_sql = ", ".join([id_col] + compare_cols + ["Date_Debut", "Actif", "Type_Changement"])
            placeholders = ", ".join(["?"] * (1 + len(compare_cols) + 3))
            values = [row[id_col]] + [row[col] for col in compare_cols] + [today, 1, 'INSERT']

            cursor.execute(f'''
                INSERT INTO {table} ({cols_sql})
                VALUES ({placeholders})
            ''', *values)
        else:
            old_row = old.iloc[0]
            if any(str(row[col]) != str(old_row[col]) for col in compare_cols):
                # 🟠 Mise à jour de l'ancien
                cursor.execute(f'''
                    UPDATE {table}
                    SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                    WHERE {id_col} = ? AND Actif = 1
                ''', today, datetime.now(), row[id_col])

                # 🆕 Insertion de la nouvelle version
                cols_sql = ", ".join([id_col] + compare_cols + ["Date_Debut", "Actif", "Type_Changement"])
                placeholders = ", ".join(["?"] * (1 + len(compare_cols) + 3))
                values = [row[id_col]] + [row[col] for col in compare_cols] + [today, 1, 'UPDATE']

                cursor.execute(f'''
                    INSERT INTO {table} ({cols_sql})
                    VALUES ({placeholders})
                ''', *values)

    conn.commit()
    cursor.close()
    print(f"✔️ Table {table} traitée avec succès.")


In [4]:
for config in config_list:
    print(f"📥 Traitement de {config['table']}...")
    load_and_historize(conn, config)


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\4113538418.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\4113538418.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)


📥 Traitement de Joueurs...
✔️ Table Joueurs traitée avec succès.
📥 Traitement de Donnees_Athletiques...
✔️ Table Donnees_Athletiques traitée avec succès.
📥 Traitement de Donnees_Techniques...


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\4113538418.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)


✔️ Table Donnees_Techniques traitée avec succès.
📥 Traitement de Donnees_Tactiques...


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\4113538418.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)


...

✔️ Table Examen_Locomoteur traitée avec succès.
📥 Traitement de Examen_ORL...


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\4113538418.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)


✔️ Table Examen_ORL traitée avec succès.
📥 Traitement de Examen_Stomatologique...


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\4113538418.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)


✔️ Table Examen_Stomatologique traitée avec succès.
